# 4. Train MACEField on α-quartz SiO₂

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/04_train_sio2.ipynb)

This notebook trains a small model from the bundled 201-frame α-quartz dataset. Each frame has 72 atoms, zero applied field, and energy, force, stress, polarization, BEC, and electronic-susceptibility labels.

The training here is a short tutorial fit. The separate analysis notebooks load a bundled reference checkpoint, so they run independently without this notebook's output. Hessians are not labels in this dataset; the dielectric lesson calculates a Hessian from the model.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Inspect the labeled structures

The split is created reproducibly from the full file with a fixed seed. The source data are from the α-quartz workflow; see the data manifest for provenance and checksums.


In [ ]:
import numpy as np
from ase.io import read

SIO2_DATA = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"
frames = read(SIO2_DATA, index=":")
fields = np.asarray([a.info["REF_electric_field"] for a in frames], dtype=float)
print("frames:", len(frames))
print("atoms per frame:", sorted({len(a) for a in frames}))
print("field vectors:", np.unique(np.round(fields, 10), axis=0))
print("info labels:", sorted(k for k in frames[0].info if k.startswith("REF_")))
print("array labels:", sorted(k for k in frames[0].arrays if k.startswith("REF_")))
print("Hessian labels:", [k for k in frames[0].info if "hessian" in k.lower()])


## Run a compact training demonstration

The default is 10 epochs on a CUDA runtime and 2 epochs on CPU. Set the environment variable MACEFIELD_TUTORIAL_EPOCHS to change it. Longer runs with validation-based patience are needed for a model-quality study.


In [ ]:
import os
import time
import yaml
import torch
import subprocess
import sys

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_EPOCHS = 10 if DEVICE == "cuda" else 2
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", DEFAULT_EPOCHS))
RUN_DIR = TUTORIAL_DIR / "runs" / f"sio2_from_scratch_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "name": "MACEField-alpha-quartz-tutorial",
    "model": "MACEField",
    "loss": "universal_field",
    "train_file": str(SIO2_DATA),
    "valid_fraction": 0.20,
    "E0s": "average",
    "seed": 23,
    "energy_key": "REF_energy",
    "forces_key": "REF_forces",
    "stress_key": "REF_stress",
    "electric_field_key": "REF_electric_field",
    "polarization_key": "REF_polarization",
    "becs_key": "REF_becs",
    "polarizability_key": "REF_polarizability",
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "energy_weight": 1.0,
    "forces_weight": 50.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 100.0,
    "polarizability_weight": 200.0,
    "num_channels": 32,
    "num_interactions": 2,
    "max_L": 1,
    "r_max": 5.0,
    "batch_size": 1,
    "valid_batch_size": 1,
    "num_workers": 0,
    "lr": 0.005,
    "max_num_epochs": EPOCHS,
    "patience": max(5, EPOCHS),
    "eval_interval": 1,
    "default_dtype": "float32",
    "device": DEVICE,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG = RUN_DIR / "train.yml"
CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print("device:", DEVICE, "| epochs:", EPOCHS)
print("configuration:", CONFIG)

subprocess.run(
    [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
    cwd=PACKAGE_REPO,
    check=True,
)


## What this fit does and does not show

This example shows the MACEField data keys, response losses, and validation split. A short run is not a converged SiO₂ potential, and its checkpoint is not used in the later lessons. The bundled checkpoint in those lessons was trained for the α-quartz workflow.
